In [34]:
import torch
import torch.nn as nn

In [35]:
block_size = 8 # n-1 gram
embedding_dim = 16
layer_size = 512
batch_size = 512
steps = 20000

In [36]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[block_size:]], dtype=torch.long)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_val = torch.tensor([ctoi[ch] for ch in val[block_size:]], dtype=torch.long)

In [37]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, block_size, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(vocab_size, embedding_dim),
            nn.Flatten(),
            nn.Linear(in_features=embedding_dim * block_size, out_features=layer_size),
            nn.ReLU(),
            nn.Linear(in_features=layer_size, out_features=vocab_size)
        )

    def forward(self, input):
        return self.layers(input)

In [38]:
# model initialization
model = MultiLayerPerceptron(block_size, vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [39]:
# training
for step in range(steps):
    optimizer.zero_grad()

    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train, y_train[ix])

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val, y_val)
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.198372840881348, val loss = 4.194258689880371
Step 1000: train loss = 1.7329719066619873, val loss = 1.9104682207107544
Step 2000: train loss = 1.5596764087677002, val loss = 1.8363968133926392
Step 3000: train loss = 1.4601829051971436, val loss = 1.7923425436019897
Step 4000: train loss = 1.6080241203308105, val loss = 1.7773187160491943
Step 5000: train loss = 1.4817034006118774, val loss = 1.7707841396331787
Step 6000: train loss = 1.44536292552948, val loss = 1.7684320211410522
Step 7000: train loss = 1.536299228668213, val loss = 1.7583779096603394
Step 8000: train loss = 1.4767484664916992, val loss = 1.7511768341064453
Step 9000: train loss = 1.5016340017318726, val loss = 1.7430001497268677
Step 10000: train loss = 1.4356846809387207, val loss = 1.7380484342575073
Step 11000: train loss = 1.5624653100967407, val loss = 1.7511258125305176
Step 12000: train loss = 1.5205776691436768, val loss = 1.7536855936050415
Step 13000: train loss = 1.5144916772842407

In [40]:
# prediction
with torch.inference_mode():
    output = ['\n'] * block_size
    max_output_length = 1000
    while len(output) < max_output_length:
        i = torch.tensor([[ctoi[ch] for ch in output[-block_size:]]], dtype=torch.long)
        
        logits = model(i)
        P = torch.softmax(logits, dim=1)

        j = torch.multinomial(P, num_samples=1, replacement=True).item()

        output.append(itoc[j])

In [41]:
print(''.join(output[block_size:]))

Cur way my two of jace, And on jices uith
Which at'turad what thou powersing, but me were e'tive con,
Or my such ander
About, my little place, is own scapt.
Give me loves no long hyisthat hand,
I'll to thward's right jorour!

MENENIUS:
Ho! I know, you be at will not how minit ye aboth. Hat is make unnorthought falsty day.
Or, kill him.

LADY CAPULET:
Halb house in of my sim this fellows;
What's that I amarefts of these whom sours;
Or toof'd fooles; to
speak, some and lets unation
husb and To peace! I can on mish him,
And by my love;
No' thither.

GLOUCESTER:
With speople's good,
I call stick and rusumpossember:
Lour I thought,
Nor speak to were, now is sword,
Was dorn.
But, but with neither to substage
theu cing
Whilst I will so sword:
Bulk'd by your achip, if thou was the her. What every
Gight,
Altist him afoles;
That to lited thus blood;
Whrail'd, my all pious to his hangrace; but ince,
And way command, my Lewis, the new best,
May you thigh eyes,
For did may ne'er say her vif
